# KR-SBERT 판례 임베딩 만들기 (앱 assets용)

`snunlp/KR-SBERT-V40K-klueNLI-augSTS` 모델로 **판례 임베딩 파일 `law_cases_krsbert.bin`** 을 만듭니다.
→ `android/app/src/main/assets/` 에 넣으세요.

- 질문용 모델(`kr_sbert_qint8.onnx`)과 단어장(`vocab.txt`)은 **이미 앱 assets에 들어 있어서** 여기서 만들지 않습니다.
  (Colab 최신 라이브러리에서는 ONNX 변환 결과가 틀어져서, 검증된 파일을 그대로 씁니다.)
- 런타임 유형을 **GPU(T4)** 로 바꾸고 위에서부터 순서대로 실행하세요.
- 판례 DB 경로·테이블·컬럼은 **1번 설정 셀**에서 바꾸면 됩니다.
- `id`는 서버(`api/lawcases/text`)가 쓰는 판례 ID와 같아야 합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

라이브러리 버전을 앱 모델을 만들 때와 똑같이 맞춥니다.
설치 후 **"세션 다시 시작"** 창이 뜨면 다시 시작하고, 이 셀 다음 셀부터 이어서 실행하세요.

In [ ]:
!pip install -q "transformers==4.49.0" "sentence-transformers==3.4.1"

## 1. 설정

In [ ]:
import os

# ▼ 판례 데이터 위치 (12만 건 데이터셋으로 바꾸면 여기만 수정)
DB_PATH  = "/content/drive/MyDrive/졸프데이터/LawData.db"
TABLE    = "legal_data"
ID_COL   = "id"        # 서버와 같은 판례 ID
TEXT_COL = "content"   # 임베딩할 본문(요약)

# ▼ 결과 저장 폴더 (구글 드라이브)
OUT_DIR = "/content/drive/MyDrive/졸프데이터/krsbert_assets"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"
BIN_OUT    = os.path.join(OUT_DIR, "law_cases_krsbert.bin")

## 2. 모델 불러오기 + 검증

앱의 ONNX 모델을 만들 때 쓴 원본 모델과 **같은 벡터가 나오는지** 확인합니다.
✅가 나와야 앱 검색 결과가 맞습니다. ❌면 이 출력을 그대로 캡처해서 공유해주세요.

In [ ]:
import numpy as np
import torch
import transformers, sentence_transformers
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print("디바이스:", device, "| transformers", transformers.__version__,
      "| sentence-transformers", sentence_transformers.__version__)
print("최대 토큰 길이:", model.max_seq_length, "| 차원:", model.get_sentence_embedding_dimension())

# 앱 모델을 만든 환경에서 계산한 기준값 (정규화된 벡터의 앞 8칸)
REFERENCE = {
    "친구가 돈을 빌려가서 연락이 안 되는데 이거 처벌 가능한가요?":
        [0.02355, -0.021, -0.04905, 0.01828, -0.02468, -0.01263, 0.01461, -0.02252],
    "형법 제347조(사기) 위반":
        [-0.01653, 0.0069, 0.00027, 0.04159, -0.00552, 0.0157, -0.08166, -0.03391],
}
ok = True
for text, expected in REFERENCE.items():
    got = model.encode(text, normalize_embeddings=True)[:8]
    diff = float(np.abs(got - np.array(expected)).max())
    print(f"차이 {diff:.5f}  {text}")
    ok &= diff < 2e-3
assert ok, "❌ 원본 모델 결과가 기준과 다릅니다. 이 출력을 그대로 캡처해서 공유해주세요."
print("✅ 모델 정상 (앱 ONNX 모델과 같은 벡터)")

## 3. 판례 불러오기

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query(f"SELECT {ID_COL} AS id, {TEXT_COL} AS text FROM {TABLE}", conn)
conn.close()

df = df.dropna(subset=["text"])
df = df[df["text"].str.strip() != ""].reset_index(drop=True)
assert df["id"].is_unique, "판례 ID가 중복됩니다"
print(f"판례 {len(df):,}건")
df.head()

## 4. 판례 임베딩 → 바이너리 저장

- `normalize_embeddings=True`: 앱의 질문 벡터도 정규화하므로 판례도 똑같이 맞춥니다.
- 파일 형식 (little-endian, 앱 `LawCaseImporter`와 같음)
  - 헤더 16바이트: `SLAW` | 형식 버전 1 | 판례 수 | 차원 768
  - 판례마다: id(int64) + 임베딩(float32 × 768)

In [ ]:
import struct
import time

start = time.time()
embeddings = model.encode(
    df["text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("<f4")
print(f"임베딩 완료: {embeddings.shape}, {time.time() - start:.0f}초")

dim = embeddings.shape[1]
records = np.empty(len(df), dtype=[("id", "<i8"), ("emb", "<f4", (dim,))])
records["id"] = df["id"].astype("int64").values
records["emb"] = embeddings

with open(BIN_OUT, "wb") as f:
    f.write(b"SLAW")
    f.write(struct.pack("<iii", 1, len(records), dim))
    records.tofile(f)

print(f"저장: {BIN_OUT} ({os.path.getsize(BIN_OUT)/1e6:.0f}MB)")

## 5. 검색 테스트 (저장한 파일을 다시 읽어서)

관련 판례가 위에 나오고 1등 점수가 0.4 이상이면 정상입니다.

In [ ]:
with open(BIN_OUT, "rb") as f:
    magic, (ver, n, d) = f.read(4), struct.unpack("<iii", f.read(12))
    loaded = np.fromfile(f, dtype=[("id", "<i8"), ("emb", "<f4", (d,))])
assert magic == b"SLAW" and n == len(loaded) == len(df)
print("판례 벡터 길이(1이어야 정상):", np.linalg.norm(loaded["emb"][:3], axis=1).round(3))

id_to_text = dict(zip(df["id"], df["text"]))

for question in [
    "친구가 돈을 빌려가서 연락이 안 되는데 이거 처벌 가능한가요?",
    "보이스피싱 피해를 당했어요. 어떻게 해야 하나요?",
    "술 마시고 운전하다 걸렸어요",
]:
    q = model.encode(question, normalize_embeddings=True)
    scores = loaded["emb"] @ q
    print(f"\n질문: {question}")
    for i in np.argsort(-scores)[:5]:
        cid = int(loaded["id"][i])
        print(f"  [{scores[i]:.3f}] id={cid}  {id_to_text[cid][:100]}...")

## 6. 앱에 넣기

드라이브 `졸프데이터/krsbert_assets/law_cases_krsbert.bin` 을 내려받아
`android/app/src/main/assets/` 에 넣으세요.

- 앱을 처음 실행하면 예전 384차원 DB를 비우고 새 판례로 다시 채웁니다.
- 예전에 이 폴더에 생긴 `kr_sbert_qint8.onnx`, `vocab.txt`는 쓰지 않으니 지워도 됩니다.